In [3]:
from transformer import TransformerLM
from transformer.optimizer import lr_cosine_schedule
from pathlib import Path
from config import get_git_commit

import torch

dataset_name = "tinystories"
tokenizer_uid = "tinystories-bpe-5k"

config = {
    "description": "tinystories_",
    "seed": 0,
    "model_class": TransformerLM,
    "model_params": {
        "vocab_size": 1000,
        "context_length": 512,
        "num_layers": 4,
        "d_model": 512,
        "d_ff": 1344,
        "num_heads": 16,
        "rope_theta": 10000,
        "device": "cuda",
        "dtype": None,  # TODO: track this down - does this control all machine precision downstream?
    },
    "optimizer_class": torch.optim.AdamW,
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": lr_cosine_schedule,
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 1000,
    },
    "training": {
        "total_steps": 1000,
        "batch_size": 128,
        "train_path": f"data/{dataset_name}/bin/{tokenizer_uid}/train.bin",
        "valid_path": f"data/{dataset_name}/bin/{tokenizer_uid}/valid.bin",
        "val_every": 10,
        "save_every": 500,
        "gpu_check_every": 50,
        "max_norm": 1.0,  # gradient-clipping threshold, passed to clip_grad_norm_(max_norm=...)
    },
    # Sources by uid in the catalog (sources.yaml), not paths -- where a bin
    # lands depends on the encoder. Overlap between the two sides, or a valid
    # source in fit_sources below, is rejected before anything is built.
    "sources": {
        "train_sources": ["romeojuliet", "odyssey"],
        "valid_sources": ["montecristo", "mobydick"],
    },
    # The encoder as a definition, not a name to look up: this block is hashed
    # into the encoder_uid that names its directory, so an edit here builds
    # beside the old one. `kind` picks an implementation from encoders.py and
    # `params` is passed to it as keyword arguments.
    "encoder": {
        "kind": "bpe",
        "params": {
            "vocab_size": 1000,
            "special_tokens": ["<|endoftext|>", "<|begin|>", "<|end|>"],
        },
        "fit_sources": ["odyssey"],
    },
    # wandb run metadata -- passed straight to wandb.init (project/name/notes/tags are
    # the caller's choice, not run_training's). Grouped under one key so the whole run
    # definition, including how it surfaces in W&B, travels as one config. `name` and
    # `tags` are overridden per run in the sweep below, same as `description`.
    "metadata": {
        "git_commit": get_git_commit(True),
        "project": "llm-pretraining",
        "name": "tinystories_",  # display name; mirror `description` per run
        "notes": "",  # free-text description, like a commit message
        "tags": [dataset_name, tokenizer_uid],  # must be a list of strings
    },
}


In [ ]:
# This run's request: the `sources` and `encoder` blocks above, verbatim, plus
# the encoder's derived address, written to runs/{RUN_ID}/run.yaml. It is all the
# Snakefile reads about the run -- the DAG falls out of it.
import sys

from config import PROJECT_ROOT

PIPELINE = PROJECT_ROOT / "notebooks" / "pipeline"
DATA = PIPELINE / "data"
sys.path.insert(0, str(PIPELINE))  # etl.py sits beside the Snakefile that shares it
import etl

# Which run this is. A fixed name for interactive work, so re-running these cells
# rewrites one request instead of littering runs/; on Modal it is the timestamped
# run_id `launch` mints. Nothing is duplicated either way -- artifacts are
# addressed by uid under data/, shared by every run that asks for them.
RUN_ID = "local"
RDIR = etl.run_dir(PIPELINE, RUN_ID)

# The embedding table is sized here and the vocabulary is fit there; disagreeing
# is silent.
assert config["model_params"]["vocab_size"] == config["encoder"]["params"]["vocab_size"]

# Catalog is workflow-level (what a uid means), request is run-level (what this
# run wants built).
spec = etl.spec(config)
etl.validate(spec, etl.read_catalog(PIPELINE))  # unknown uids, train/valid overlap, leaked holdout
etl.write_spec(spec, RDIR)

encoder_uid = spec["encoder"]["encoder_uid"]
print(f"run      {RDIR.relative_to(PIPELINE)}")
print(f"encoder  {etl.encoder_artifact(DATA, encoder_uid).relative_to(PIPELINE)}")
for side in ("train_sources", "valid_sources"):
    for uid in spec["sources"][side]:
        print(f"{side[:5]:>8}  {etl.encoded_bin(DATA, encoder_uid, uid).relative_to(PIPELINE)}")


In [ ]:
# What this run's run.yaml asks for, without building it.
#
# sys.executable because the shell's PATH is not necessarily this kernel's env;
# cores and flags come from profiles/default/config.yaml beside the Snakefile.
# --config goes last here and below: it takes a variable-length list of key=value
# pairs, so anything non-dash following it is swallowed as another pair.
!{sys.executable} -m snakemake -s {PIPELINE}/Snakefile -d {PIPELINE} -n --config run_id={RUN_ID}


In [ ]:
# Build what is missing. Encoders and bins are write-once: an existing one is
# never rebuilt, not even after its source changes, so a source edit only shows
# up in bins created after it.
#
# To build one piece, add a target -- `encoder`, `train`, `valid`, or a source uid
# -- before the options, not after --config.
!{sys.executable} -m snakemake -s {PIPELINE}/Snakefile -d {PIPELINE} --config run_id={RUN_ID}


In [ ]:
# The graph as snakemake resolved it; jobs already done are dashed.
from IPython.display import SVG, display

!{sys.executable} -m snakemake -s {PIPELINE}/Snakefile -d {PIPELINE} --dag --config run_id={RUN_ID} 2>/dev/null | dot -Tsvg > {PIPELINE}/dag.svg
display(SVG(filename=f"{PIPELINE}/dag.svg"))
